# AI-Powered Energy Consumption Forecasting - end-to-end walkthrough
Run the cells top to bottom. Each section matches a phase in the README.
(Run `python main.py` once from the project root first if you want the saved model and charts.)

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))      # make the src/ package importable
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
from src import config
from src.data_generation import generate_energy_data
from src.data_loader import clean_data
from src.features import make_supervised
from src.train import chronological_split, train_all
from src.forecast import recursive_forecast

## Phase 2 - Look at the raw data (messy on purpose)

In [ ]:
raw = generate_energy_data()
print(raw.shape, "| missing Energy values:", raw["Energy"].isna().sum())
raw.head(10)

## Phase 3 - Cleaning

In [ ]:
clean, report = clean_data(raw)
print(report)
clean["Energy"].plot(figsize=(14,4), lw=0.3, title="Cleaned hourly consumption (kWh)");

## Phase 4 - Feature engineering
Calendar + weather + *day-ahead-safe* lag features (never lag_1!).

In [ ]:
X, y = make_supervised(clean)
X_train, X_test, y_train, y_test = chronological_split(X, y)
print(X.shape); X.head()

## Phase 5 & 6 - Train every model and compare on the unseen future

In [ ]:
metrics, fitted, preds, specs = train_all(X_train, X_test, y_train, y_test)
metrics

In [ ]:
best = metrics.drop(index="Seasonal Naive (last week)").index[0]
print("Best model:", best)
plt.figure(figsize=(14,4))
plt.plot(y_test.index[-168:], y_test.iloc[-168:], label="Actual")
plt.plot(y_test.index[-168:], preds[best].iloc[-168:], "--", label="Predicted")
plt.legend(); plt.title("Last 7 days of the test set"); plt.show()

## Phase 7 - Forecast the next 7 days

In [ ]:
model, cols = specs[best]
future = recursive_forecast(model, cols, clean, horizon=168, residual_std=(y_test-preds[best]).std())
ax = clean["Energy"].iloc[-168:].plot(figsize=(14,4), label="recent actual")
future["Forecast"].plot(ax=ax, label="forecast")
ax.fill_between(future.index, future["Lower"], future["Upper"], alpha=0.2)
ax.legend(); future.head()